In [1]:
import re
import os
import json
import numpy
import collections

import kagglehub
import pandas as pd
from nltk.corpus import stopwords

/home/tdp/Desktop/repos/fiuba/distribuidos/tp-diseno/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Needed for the French language stopwords and advanced processing
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /home/tdp/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
/home/tdp/Desktop/repos/fiuba/distribuidos/tp-diseno/.venv/lib/python3.12/site-packages/nltk/downloader.py:717: UserWarning: NLTK will not authorize the non-private download directory '/home/tdp/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(item, download_dir, force):
[nltk_data]    | Downloading package alpino to /home/tdp/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/tdp/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/tdp/nltk_data...
[nlt

True

In [18]:
KAGGLE_DATASET_BASE_PATH = "wikimedia-foundation/wikipedia-structured-contents"
KAGGLE_EN_FILE_BASE_PATH = "enwiki/data/enwiki_namespace_0_"
KAGGLE_FR_FILE_BASE_PATH = "frwiki/data/frwiki_namespace_0_"
PER_LANGUAGE_SAMPLE_AMOUNT = 1

COLUMNS_TO_DROP = ['additional_entities', 'date_created',
       'description', 'event', 'identifier',
       'infoboxes', 'is_part_of', 'license', 'main_entity',
       'references', 'tables', 'version']

wiki_entry_df = None
for kaggle_language_file_path in [KAGGLE_EN_FILE_BASE_PATH, KAGGLE_FR_FILE_BASE_PATH]:
    for i in range(PER_LANGUAGE_SAMPLE_AMOUNT):
        kaggle_file_path = kaggle_language_file_path + f"{i:05}.parquet"
        dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_BASE_PATH, path=kaggle_file_path)
        
        df = pd.read_parquet(dataset_path)
        print(f"Df Columns: {df.columns}")
        df = df.drop(columns=COLUMNS_TO_DROP)
        if wiki_entry_df is None:
            wiki_entry_df = df
            continue
        wiki_entry_df = pd.concat([wiki_entry_df, df])

print(f"Final DataFrame Columns: {wiki_entry_df.columns}")
wiki_entry_df.info()

Df Columns: Index(['abstract', 'additional_entities', 'date_created', 'date_modified',
       'description', 'event', 'identifier', 'image', 'in_language',
       'infoboxes', 'is_part_of', 'license', 'main_entity', 'name',
       'references', 'sections', 'tables', 'url', 'version'],
      dtype='str')
Df Columns: Index(['abstract', 'additional_entities', 'date_created', 'date_modified',
       'description', 'event', 'identifier', 'image', 'in_language',
       'infoboxes', 'is_part_of', 'license', 'main_entity', 'name',
       'references', 'sections', 'tables', 'url', 'version'],
      dtype='str')
Final DataFrame Columns: Index(['abstract', 'date_modified', 'image', 'in_language', 'name', 'sections',
       'url'],
      dtype='str')
<class 'pandas.DataFrame'>
Index: 50000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   abstract       49771 non-null  str           
 1   d

In [19]:
wiki_entry_df.head()

,abstract,date_modified,image,in_language,name,sections,url
0,"""Khan gizi"" spring is a spring located next to...",2024-10-31 12:58:40,{'content_url': 'https://upload.wikimedia.org/...,{'identifier': 'en'},"""Khan gizi"" spring","[{""type"":""section"",""name"":""Abstract"",""has_part...",https://en.wikipedia.org/wiki/%22Khan_gizi%22_...
1,Events during the year 1113 in Italy.,2024-10-07 13:20:34,None,{'identifier': 'en'},1113 in Italy,"[{""type"":""section"",""name"":""Abstract"",""has_part...",https://en.wikipedia.org/wiki/1113_in_Italy
2,The 18th Infantry Division was a French Army f...,2024-10-24 09:29:32,None,{'identifier': 'en'},18th Infantry Division (France),"[{""type"":""section"",""name"":""Abstract"",""has_part...",https://en.wikipedia.org/wiki/18th_Infantry_Di...
3,Parliamentary elections were held in Greece on...,2025-01-27 21:55:24,None,{'identifier': 'en'},1933 Greek parliamentary election,"[{""type"":""section"",""name"":""Abstract"",""has_part...",https://en.wikipedia.org/wiki/1933_Greek_parli...
4,The 1903 German football championship was the ...,2024-01-01 23:47:26,None,{'identifier': 'en'},1903 German football championship,"[{""type"":""section"",""name"":""Abstract"",""has_part...",https://en.wikipedia.org/wiki/1903_German_foot...


In [23]:
print(wiki_entry_df["image"][0])
#wiki_entry_df["date_modified"].head(1)

0    {'content_url': 'https://upload.wikimedia.org/...
0                                                 None
Name: image, dtype: object


In [6]:
# Dataset cleanup
def in_language_parse(in_language_column):
    if not isinstance(in_language_column, dict):
        return None
    return in_language_column.get('identifier', None)
wiki_entry_df["in_language"] = wiki_entry_df["in_language"].apply(in_language_parse)

def image_object_to_url(image):
    if not isinstance(image, dict):
        return None
    return image.get('content_url', None)
wiki_entry_df["image"] = wiki_entry_df["image"].apply(image_object_to_url)

In [7]:
# Entries modified before or at 2020
wiki_entry_df['date_modified_year'] = wiki_entry_df['date_modified'].dt.year
wiki_entry_before_2020_df = wiki_entry_df[wiki_entry_df['date_modified_year'] <= 2020].copy()

In [8]:
# Url and date of modification of english entries before 2020
wiki_entry_before_2020_df[wiki_entry_before_2020_df["in_language"] == "en"][["date_modified", "url"]]

,date_modified,url
19,2019-08-16 05:54:47,https://en.wikipedia.org/wiki/18th_Infantry_Br...
28,2019-03-17 20:10:31,https://en.wikipedia.org/wiki/122nd_Division
63,2019-02-04 18:04:25,https://en.wikipedia.org/wiki/105th_Squadron_(...
75,2014-05-20 14:10:52,https://en.wikipedia.org/wiki/160th_meridian
81,2020-06-11 11:02:31,https://en.wikipedia.org/wiki/1878%E2%80%9379_...
...,...,...
24891,2018-03-14 13:14:37,https://en.wikipedia.org/wiki/Albert_Bridge
24933,2019-01-09 18:51:36,https://en.wikipedia.org/wiki/AFDB
24945,2014-10-10 17:17:46,https://en.wikipedia.org/wiki/126th_Division
24948,2018-04-10 11:16:34,https://en.wikipedia.org/wiki/2004_World_Inter...


In [9]:
# Name, date of modification and image of all articles that were modified before 2020 and have a biography/biographie section
BIOGRAPHY_SECTION_NAMES = ['biographie', 'biography']
def has_biography_section(sections):
    if not isinstance(sections, str):
        return False
    section_names =  [section.get("name", "").lower() for section in json.loads(sections)]
    for biography_section_name in BIOGRAPHY_SECTION_NAMES:
        if biography_section_name in section_names:
            return True
    return False
wiki_entry_before_2020_df['has_biography_section'] = wiki_entry_before_2020_df['sections'].apply(has_biography_section)

#Note: Actual implementation should download the image from the url
wiki_entry_before_2020_df[(wiki_entry_before_2020_df['has_biography_section'] == True)][["name", "date_modified", "image"]]

,name,date_modified,image
13678,Albert Samuel Gatschet,2019-06-20 21:56:57,https://upload.wikimedia.org/wikipedia/commons...
16743,Alexandre Fiodorovitch Plechtcheïev,2017-03-07 09:53:08,https://upload.wikimedia.org/wikipedia/commons...
16756,Adam McGeorge,2019-10-25 16:09:05,NaN
16988,Attilio Perez,2020-08-02 09:59:12,https://upload.wikimedia.org/wikipedia/commons...
17498,Andrés Duarte,2019-11-09 20:11:22,NaN
17687,Aaron Downes,2020-11-11 03:15:18,NaN
17949,Antoni Łyko,2019-11-03 15:09:38,https://upload.wikimedia.org/wikipedia/commons...
18131,Alison Kreviazuk,2019-05-14 17:56:45,NaN
18197,Abebaw Butako,2019-10-29 00:30:06,NaN
18526,Alfred Fitch,2020-12-10 21:18:23,NaN


In [10]:
# Word and count of the 20 most common words in the abstract
TOP_SIZE = 20
WORD_REGEX = r'\b\w+\b'
NUMBER_REGEX = r'[+-]?((\d+\.?\d*)|(\.\d+))'
# Source and discussion: https://stackoverflow.com/questions/12643009/regular-expression-for-floating-point-numbers/42629198#42629198

def extract_non_stopwords_from_abstract(abstract):
    if not isinstance(abstract, str):
        return []
    words = re.findall(WORD_REGEX, abstract.lower())
    return [word for word in words if word not in stop_words and not re.fullmatch(NUMBER_REGEX, word)]

stop_words = set(stopwords.words('english')) | set(stopwords.words('french'))
words_counter = collections.Counter()
for abstract in wiki_entry_df['abstract'].dropna():
    non_stop_words = extract_non_stopwords_from_abstract(abstract)
    words_counter.update(set(non_stop_words))
pd.DataFrame(words_counter.most_common(TOP_SIZE), columns=["word", "count"])

LookupError: 
**********************************************************************
  Resource 'stopwords' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('stopwords')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'corpora/stopwords'

  Searched in:
    - '/home/tdp/nltk_data'
    - '/home/tdp/Desktop/repos/fiuba/distribuidos/tp-diseno/.venv/nltk_data'
    - '/home/tdp/Desktop/repos/fiuba/distribuidos/tp-diseno/.venv/share/nltk_data'
    - '/home/tdp/Desktop/repos/fiuba/distribuidos/tp-diseno/.venv/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [10]:
# Shared words between all french and english articles in the abstract
def fill_language_word_set(df, language):
    language_words = set()
    for abstract in df[df["in_language"] == language]['abstract'].dropna():
        language_words |= set(extract_non_stopwords_from_abstract(abstract))
    return language_words

english_words = fill_language_word_set(wiki_entry_df, "en")
french_words = fill_language_word_set(wiki_entry_df, "fr")
shared_words = english_words & french_words
pd.Series(data=list(english_words & french_words))

0               pécsi
1              burney
2            incertum
3             library
4           pilhatsch
             ...     
57428     engagements
57429        baltasar
57430         salcedo
57431    californicus
57432            cult
Length: 57433, dtype: object

In [11]:
# Entries after 2020
wiki_entry_after_2020_df = wiki_entry_df[wiki_entry_df['date_modified_year'] > 2020].copy()

In [12]:
# Url and count of all articles modified after 2020 with an abstract word count greater than the average of all articles written before or at 2020
def calc_abstract_word_count_average(df):
    count = 0
    total = 0
    for abstract in df['abstract'].dropna():
        total += len(extract_non_stopwords_from_abstract(abstract))
        count += 1
    return total/count

before_2020_abstract_average = calc_abstract_word_count_average(wiki_entry_before_2020_df)
print("Before 2020 abstract average:", before_2020_abstract_average)

Before 2020 abstract average: 23.20417858354743


In [13]:
wiki_entry_after_2020_df["abstract_word_count"] = wiki_entry_after_2020_df["abstract"].apply(lambda abstract: len(extract_non_stopwords_from_abstract(abstract)))
wiki_entry_after_2020_df[wiki_entry_after_2020_df["abstract_word_count"] > before_2020_abstract_average][["url","abstract_word_count"]]

,url,abstract_word_count
0,https://en.wikipedia.org/wiki/%22Khan_gizi%22_...,58
3,https://en.wikipedia.org/wiki/1933_Greek_parli...,42
4,https://en.wikipedia.org/wiki/1903_German_foot...,80
8,https://en.wikipedia.org/wiki/1892_Rangitikei_...,52
11,https://en.wikipedia.org/wiki/12th_Internation...,31
...,...,...
24990,https://fr.wikipedia.org/wiki/Breakdown_(jeu_v...,32
24991,https://fr.wikipedia.org/wiki/Acide_docosapent...,40
24997,https://fr.wikipedia.org/wiki/Alton_United_Foo...,30
24998,https://fr.wikipedia.org/wiki/Constitution_col...,24
